# Field-Weighted Citation Impact (FWCI)
FWCI is the primary impact metric in the manuscript: it normalizes citation performance against publications from the same field and publication year, so it needs no separate adjustment for article age or field.

All comparisons are run at two levels: (1) a one-sided comparison of sharing vs. non-sharing articles, and (2) a two-sided comparison across the three sharing granularities (FIXATION, TRIAL, PARTICIPANT).

In [1]:
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from helpers.config import *
from helpers import dataset
from helpers.plotting import save_figure

pio.renderers.default = "browser"

In [2]:
combined, FEATURES_DF = dataset.load_or_build()

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\.claude\worktrees\eye-movement-analysis-review-22482b\data_store (234 rows)


### (1) FWCI
#### (1A) Sharing / Non-Sharing Articles

In [3]:
not_sharing_fwci = combined.loc[~combined["is_sharing_data"], "FieldWeightedCitationIndex"]
sharing_fwci = combined.loc[combined["is_sharing_data"], "FieldWeightedCitationIndex"]

# check normality of FWCI distributions:
print("Is FWCI normally distributed for non-sharing articles?")
fwci_non_sharing_shapiro = stats.shapiro(not_sharing_fwci)
print(f"{"YES" if fwci_non_sharing_shapiro.pvalue > 0.05 else "NO"} (W={fwci_non_sharing_shapiro.statistic:.3f}, p={fwci_non_sharing_shapiro.pvalue:.4f})\n")

print("Is FWCI normally distributed for sharing articles?")
fwci_sharing_shapiro = stats.shapiro(sharing_fwci)
print(f"{"YES" if fwci_sharing_shapiro.pvalue > 0.05 else "NO"} (W={fwci_sharing_shapiro.statistic:.3f}, p={fwci_sharing_shapiro.pvalue:.4f})")

Is FWCI normally distributed for non-sharing articles?
NO (W=0.833, p=0.0000)

Is FWCI normally distributed for sharing articles?
NO (W=0.880, p=0.0000)


In [4]:
fwci_pair_test = stats.mannwhitneyu(not_sharing_fwci, sharing_fwci, alternative="less")

# calculate effect size (rank-biserial correlation / CLES)
n1, n2 = len(not_sharing_fwci), len(sharing_fwci)
U_nonshare = fwci_pair_test.statistic
U_share = n1 * n2 - U_nonshare
rank_biserial_corr = (U_share - U_nonshare) / (n1 * n2)
cles = U_share / (n1 * n2)

print("Share / Non-Share FWCI Comparison:")
print("Mann-Whitney U Test:" + f"\tU={fwci_pair_test.statistic},\tp={fwci_pair_test.pvalue:.4f}")
print("{0: <20}".format("Effect Sizes:") + f"\tRBC: {rank_biserial_corr:.3f}\tCLES: {cles:.3f}")

# re-run statistical test with `pingouin` for robustness:
pg.mwu(not_sharing_fwci, sharing_fwci, alternative="less")

Share / Non-Share FWCI Comparison:
Mann-Whitney U Test:	U=5321.0,	p=0.0439
Effect Sizes:       	RBC: 0.136	CLES: 0.568


,U_val,alternative,p_val,RBC,CLES
MWU,5321.0,less,0.043902,-0.136201,0.568101


#### (1B) Data Sharing Class

In [5]:
# check normality of FWCI distributions for each sharing class:
share_group_fwci = []
for share_group in ["FIXATION", "TRIAL", "PARTICIPANT"]:
    group_fwci = (
        combined
        .loc[combined["data_sharing_class"] == share_group, "FieldWeightedCitationIndex"]
        .dropna()
        .reset_index(drop=True)
        .rename(share_group)
    )
    share_group_fwci.append(group_fwci)
    shapiro_result = stats.shapiro(group_fwci)
    print(f"Is FWCI normally distributed for {share_group} sharing articles?")
    print(f"{"YES" if shapiro_result.pvalue > 0.05 else "NO"} (W={shapiro_result.statistic:.3f}, p={shapiro_result.pvalue:.4f})\n")

# run Kruskal-Wallis H-test:
fwci_sharegroup_test = stats.kruskal(*share_group_fwci)
fwci_sharegroup_test

Is FWCI normally distributed for FIXATION sharing articles?
NO (W=0.832, p=0.0002)

Is FWCI normally distributed for TRIAL sharing articles?
NO (W=0.842, p=0.0010)

Is FWCI normally distributed for PARTICIPANT sharing articles?
YES (W=0.939, p=0.1753)



KruskalResult(statistic=np.float64(3.0816659719664665), pvalue=np.float64(0.21420259932961308))

#### Visualizing Share/Non-Share FWCI Comparison

In [6]:
suplot_titles = ["Sharing vs. Non-Sharing", "Different Types of Data Sharing"]
fwci_fig = make_subplots(
    rows=1, cols=len(suplot_titles), column_titles=suplot_titles,
    shared_yaxes=True, horizontal_spacing=0.025, vertical_spacing=0.025,
)

# (1) Sharing vs. Non-Sharing
left_subplot_data_long = (
    combined
    .melt(
        id_vars=['is_sharing_data'],
        value_vars=['FieldWeightedCitationIndex'],
        var_name='name',
        value_name='value'
    )
    .replace({
        'FieldWeightedCitationIndex': 'FWCI',
    })
)
max_y = left_subplot_data_long["value"].max()
for is_share in left_subplot_data_long["is_sharing_data"].unique():
    subset = left_subplot_data_long[left_subplot_data_long["is_sharing_data"] == is_share]
    subset_name = 'Sharing' if is_share else 'Not Sharing'
    color = CLASS_COLORS[subset_name.upper()]
    fwci_fig.add_trace(
        row=1, col=1,
        trace=go.Violin(
            x=subset['name'], y=subset['value'],
            name=subset_name, legendgroup=subset_name, scalegroup=subset_name,
            side='positive' if is_share else 'negative',
            fillcolor=color, line_color=color,
            width=0.9, spanmode="hard",
            points=False, pointpos=0, jitter=0.5,
            box=dict(visible=False, width=0.5, line=dict(color="black")),
            meanline=dict(visible=False, color='black'),
        )
    )
    # annotate half-violin
    fwci_fig.add_annotation(
        row=1, col=1,
        text=subset_name, font={**AXIS_TITLE_FONT, "color": color},
        x=0.25 if is_share else -0.2, xanchor="center", xref="x",
        y=0.925 * max_y, yanchor="middle", yref="paper",
        showarrow=False,
    )

# add significance asterisk
fwci_fig.add_annotation(
    row=1, col=1,
    x="FWCI", xanchor="center", xref="x",
    y=max_y * 1.05, yanchor="middle", yref="paper",
    text="*", font=TITLE_FONT, font_size=30,
    showarrow=False,
)
# add separation line
fwci_fig.add_shape(
    row=1, col=1,
    type="line", line=dict(color="black", width=2, dash="dash"),
    x0="FWCI", x1="FWCI", y0=0, y1=max_y,
)

# (2) Data Sharing Class
right_subplot_data_long = (
    combined
    .loc[combined["data_sharing_class"].isin(["FIXATION", "TRIAL", "PARTICIPANT"])]
    .melt(
        id_vars=['data_sharing_class'],
        value_vars=['FieldWeightedCitationIndex'],
        var_name='name',
        value_name='value'
    )
    .replace({
        'FieldWeightedCitationIndex': 'FWCI',
    })
)
for i, share_class in enumerate(right_subplot_data_long["data_sharing_class"].unique()):
    subset = right_subplot_data_long[right_subplot_data_long["data_sharing_class"] == share_class]
    color = CLASS_COLORS[share_class]
    fwci_fig.add_trace(
        row=1, col=2,
        trace=go.Violin(
            x=subset['data_sharing_class'], y=subset['value'],
            name=share_class, legendgroup=share_class, scalegroup=share_class,
            side='positive', points='all', pointpos=-0.25, jitter=0.2,
            fillcolor=color, line=dict(color=color, width=1.5),
            width=1.0, spanmode="hard",
            box=dict(visible=False, width=0.5, line=dict(color="gray")),
            meanline=dict(visible=False, color='gray'),
        ))
    # annotate half-violin
    fwci_fig.add_annotation(
        row=1, col=2,
        text=share_class.title(),
        font={**AXIS_TITLE_FONT, "color": color},
        x=i, xanchor="center", xref="x",
        y=0.925 * max_y, yanchor="middle", yref="paper",
        showarrow=False,
    )

# add n.s. mark
fwci_fig.add_annotation(
    row=1, col=2,
    x=1.0, xanchor="center", xref="x",
    y=max_y * 1.05, yanchor="middle", yref="paper",
    text="n.s.", font=TITLE_FONT,
    showarrow=False,
)


# update layout and show
for ann in fwci_fig.layout.annotations:
    if ann.text not in suplot_titles:
        continue
    ann_xref = 'x' if ann.text == suplot_titles[0] else 'x2'
    ann.update(dict(
        font=TITLE_FONT,
        x=-0.5, xanchor="left", xref=ann_xref,
        y=1.0, yanchor="top", yref="paper",
))
fwci_fig.update_xaxes(
    title=None, showticklabels=False, tickfont=AXIS_TICK_FONT, zeroline=False,
)
fwci_fig.update_yaxes(
    showgrid=True, gridcolor='lightgrey', gridwidth=1.5,
    zeroline=False,
)
fwci_fig.update_yaxes(
    row=1, col=1,
    range=[-0.5, max_y * 1.2],
    title=dict(text="<b>FWCI</b>", font=AXIS_TITLE_FONT, standoff=5),
    tickfont=AXIS_TICK_FONT,
)
fwci_fig.update_layout(
    width=1000, height=500,
    title=dict(
        text="<b>FWCI Distribution for Data Sharing Class</b>",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top"
    ),
    violinmode='overlay',
    violingap=0,
    legend=dict(visible=False,),
    margin=dict(t=50, b=5, l=50, r=10, pad=0),
    template="plotly_white",
)
fwci_fig.show()

In [7]:
# flip to True to export this figure into `output/`
if False:
    save_figure(fwci_fig, "fwci_data_sharing.png", width=1000, height=500)

### Adjusting for Other Citation Predictors
Publication age is deliberately excluded from this model: OpenAlex normalizes FWCI by field,
publication year and document type, so age is accounted for by construction.

Venue is a different case. FWCI does **not** normalize it away, so venue impact remains a real
predictor here. It is left out of the model below on purpose: sharers publish in higher-impact
venues, which makes venue a plausible *mediator* of the sharing effect rather than a nuisance to
be partialled out. The model below is therefore the unadjusted estimate, and the venue-adjusted
estimate should be reported beside it, with the difference between the two read as mechanism.

In [8]:
def build_fwci_model_df(combined: pd.DataFrame, features_df: pd.DataFrame) -> pd.DataFrame:
    """Feature matrix plus log1p FWCI, renamed for `smf.ols` via `dataset.to_smf_columns`.

    Age is excluded from the *covariates* below (not from this frame): FWCI already normalizes
    for field, publication year and document type. Venue impact is excluded for a different
    reason, as a plausible mediator rather than a confounder. See the markdown above.
    """
    log_fwci = np.log(combined["FieldWeightedCitationIndex"] + 1).rename("log(FWCI)")
    df = pd.concat([features_df, log_fwci], axis=1)
    return dataset.to_smf_columns(df)

fwci_model_df = build_fwci_model_df(combined, FEATURES_DF)

covariate_model = smf.ols(
    "log_fwci ~ is_sharing_data + is_open_access + has_preprint + has_us_author + log_number_of_authors",
    # log_weeks_since_pub is normalized into FWCI by design; venue_impact is left out as a
    # mediator, not because FWCI normalizes it (it does not)
    data=fwci_model_df,
).fit()

beta = covariate_model.params["is_sharing_data"]
ci_low, ci_high = covariate_model.conf_int().loc["is_sharing_data"]
print(
    f"\nData-sharing effect adjusted for author count, US authorship, open access, and pre-print publication"
    f"\n(publication age is normalized into FWCI by construction; venue impact is deliberately not adjusted for):"
    f"\n\tbeta = {beta:.3f}  (95% CI [{ci_low:.3f}, {ci_high:.3f}]),  p = {covariate_model.pvalues['is_sharing_data']:.3f}"
    f"\n\tmultiplicative citation advantage = exp(beta) = {np.exp(beta):.3f}× ({(np.exp(beta) - 1) * 100:+.1f}%)"
)
covariate_model.summary()


Data-sharing effect adjusted for author count, US authorship, open access, and pre-print publication
(publication age is normalized into FWCI by construction; venue impact is deliberately not adjusted for):
	beta = 0.089  (95% CI [-0.044, 0.222]),  p = 0.187
	multiplicative citation advantage = exp(beta) = 1.093× (+9.3%)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               log_fwci   R-squared:                       0.063
Model:                            OLS   Adj. R-squared:                  0.043
Method:                 Least Squares   F-statistic:                     3.079
Date:                Tue, 08 Sep 2026   Prob (F-statistic):             0.0104
Time:                        14:49:55   Log-Likelihood:                -148.25
No. Observations:                 234   AIC:                             308.5
Df Residuals:                     228   BIC:                             329.2
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
=========================================================================================
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.4691      0.115      4.096      0.000       0.243       0.695
is_sharing_data           0.0891      0.067      1.323      0.187      -0.044       0.222
is_open_access            0.1639      0.086      1.907      0.058      -0.005       0.333
has_preprint              0.1255      0.079      1.588      0.114      -0.030       0.281
has_us_author             0.1432      0.061      2.338      0.020       0.022       0.264
log_number_of_authors     0.0303      0.072      0.422      0.674      -0.111       0.172
==============================================================================
Omnibus:                       11.651   Durbin-Watson:                   2.068
Prob(Omnibus):                  0.003   Jarque-Bera (JB):               10.539
Skew:                           0.453   Prob(JB):                      0.00515
Kurtosis:                       2.490   Cond. No.                         8.40
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""